##### Import the libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

##### Load the datasets

In [ ]:
ad = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\admission_discharge_cleaned.csv")
bd = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\bed_inventory_cleaned.csv")
oa = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\outpatient_arrivals_cleaned.csv")
es = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\elective_surgery_cleaned.csv")
hr = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\hospital_reference_cleaned.csv")
sr = pd.read_csv (r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\staffing_resource_cleaned.csv")


##### Aggregate each dataset to the daily level.

In [ ]:
daily_admissions = (ad.groupby(["hospital_id", "admission_datetime"]).size().reset_index(name="total_admissions"))
daily_discharges = (ad.groupby(["hospital_id", "discharge_datetime"]).size().reset_index(name="total_discharges"))


In [ ]:
daily_beds = (bd.groupby(["hospital_id", "datetime"]).agg({"total_beds": "first", "occupied_beds": "mean", "staffed_beds": 
             "mean","closed_beds": "mean","occupancy_rate": "mean"}).reset_index())


In [ ]:
completed = es[es["surgery_status"] == "Completed"]

daily_completed_surgeries = (completed.groupby(["hospital_id", "surgery_date"]).agg(completed_surgeries=("surgery_id", "count"),
                            expected_los=("expected_los_days", "mean")).reset_index())

In [ ]:
cancelled = es[es["surgery_status"] == "Cancelled"]

daily_cancelled_surgeries = (cancelled.groupby(["hospital_id", "surgery_date"]).agg(cancelled_surgeries=("surgery_id", "count")).reset_index())

In [ ]:
daily_staffing = sr.groupby(["hospital_id", "date"]).agg(planned_staff=("planned_staff", "sum"), actual_staff=("actual_staff", "sum"),
                 safe_ratio=("safe_ratio_met", "mean")).reset_index()

In [ ]:
daily_outpatients = (oa.groupby(["hospital_id", "arrival_datetime"]).agg(arrivals=("arrival_id", "count"),
                    avg_time_to_admission=("time_to_admission", "mean"),bed_requests=("bed_requested", "sum")).reset_index())

##### Rename the date

In [ ]:
daily_admissions = daily_admissions.rename(columns={"admission_datetime": "date"})
daily_discharges = daily_discharges.rename(columns={"discharge_datetime": "date"})
daily_beds = daily_beds.rename(columns={"datetime": "date"})
daily_outpatients = daily_outpatients.rename(columns={"arrival_datetime": "date"})
daily_completed_surgeries = daily_completed_surgeries.rename(columns={"surgery_date": "date"})
daily_cancelled_surgeries = daily_cancelled_surgeries.rename(columns={"surgery_date": "date"})

In [ ]:
datasets = {
    "Admissions": daily_admissions,
    "Discharges": daily_discharges,
    "Beds": daily_beds,
    "Completed Surgeries": daily_completed_surgeries,
    "Cancelled Surgeries": daily_cancelled_surgeries,
    "Outpatients": daily_outpatients,
    "Staffing": daily_staffing,
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print(df.shape)
    print(df.head())
    print(df.info)


Admissions
----------------------------------------
(3659, 3)
  hospital_id        date  total_admissions
0  HHN-BIR-01  2024-01-01                20
1  HHN-BIR-01  2024-01-02                35
2  HHN-BIR-01  2024-01-03                29
3  HHN-BIR-01  2024-01-04                35
4  HHN-BIR-01  2024-01-05                41
<bound method DataFrame.info of      hospital_id        date  total_admissions
0     HHN-BIR-01  2024-01-01                20
1     HHN-BIR-01  2024-01-02                35
2     HHN-BIR-01  2024-01-03                29
3     HHN-BIR-01  2024-01-04                35
4     HHN-BIR-01  2024-01-05                41
...          ...         ...               ...
3654  HHN-MAN-01  2025-12-28                48
3655  HHN-MAN-01  2025-12-29                45
3656  HHN-MAN-01  2025-12-30                35
3657  HHN-MAN-01  2025-12-31                45
3658  HHN-MAN-01  2026-01-01                 5

[3659 rows x 3 columns]>

Discharges
---------------------------------------

##### Save aggregates datasets

In [ ]:
daily_admissions.to_csv("../Data/Aggregated/daily_admissions.csv", index=False)

daily_discharges.to_csv("../Data/Aggregated/daily_discharges.csv", index=False)

daily_beds.to_csv("../Data/Aggregated/daily_beds.csv", index=False)

daily_completed_surgeries.to_csv("../Data/Aggregated/daily_completed_surgeries.csv",index=False)

daily_cancelled_surgeries.to_csv("../Data/Aggregated/daily_cancelled_surgeries.csv",index=False)

daily_outpatients.to_csv("../Data/Aggregated/daily_outpatients.csv",index=False)

##### Merge everything

In [ ]:
df = daily_admissions.copy()

df = df.merge(daily_discharges, on=["hospital_id", "date"], how="left")
df = df.merge(daily_beds, on=["hospital_id", "date"], how="left")
df = df.merge(daily_completed_surgeries,on=["hospital_id", "date"], how="left")
df = df.merge(daily_cancelled_surgeries, on=["hospital_id", "date"], how="left")
df = df.merge(daily_outpatients, on=["hospital_id", "date"], how="left")
df = df.merge(daily_staffing, on=["hospital_id", "date"], how="left")
df = df.merge(hr, on="hospital_id", how="left")

##### Save merged dataset

In [ ]:
df.to_csv("../Data/Merged/hospital_bed_demand_dataset.csv", index=False)